### Week 7. Problem set
Ilias Dzhabbarov

In [ ]:
type Name = String
data Grade = A | B | C | D
data Student = Student Name Grade
data Point2D = Point2D Int Int
data Status a
    = Stop
    | Warn [String] a

Task 1

In [ ]:
dup :: (a -> a -> b) -> a -> b
dup f x = f x x

dip :: (a -> a -> a) -> a -> a
dip f x = f x (f x x)

twice :: (a -> a) -> a -> a
twice f x = f (f x)

Task 2

In [ ]:
a :: Int -> Point2D
a = dup Point2D

Answer (a): Int -> Point2D

```Haskell
Point2D :: Int -> Int -> Point2D

dup :: (a -> a -> b) -> a -> b

dup :: (Int -> Int -> Point2D) -> Int -> Point2d

dup Point2D :: Int -> Point2D
```

In [ ]:
b = dup (dup (+)) 3

: 

Answer (b): Type error

```Haskell
dup :: (a -> a -> b) -> a -> b
+ :: Int -> Int -> Int
```
So
```Haskell
dup (+) :: Int -> Int
```
also could be called `multiply2`


```Haskell
dup (dup (+))
```
`dup` expected `a -> a -> b` as an input but got `Int -> Int`


In [ ]:
c = twice dip

: 

Answer (c): Type error

Intuition: `twice` expected a fuction that has the same "input" and "return" type `(a -> a)`

Let's have a step-by-step review:

`dip` type is `(t1 -> t1 -> t1) -> t1 -> t1`

`twice` type is `(t2 -> t2) -> t2 -> t2`

so in `twice dip` it tries to match `(t2 -> t2)` (first parameter) with `(t1 -> t1 -> t1) -> t1 -> t1` (`dip` type)
```Haskell
dip :: t2 -> t2
dip :: (t1 -> t1 -> t1) -> t1 -> t1

t2 = (t1 -> t1 -> t1)
```
error when it trying to match `t2` which is `t1 -> t1 -> t1` to the `t1 -> t1`

In [ ]:
d = dip dip

: 

Answer (d): Type error

```Haskell
dip :: (a -> a -> a) -> a -> a
```
Intuition: `dip` expected a fuction that has 2 arguments with the same type as a return `(a -> a -> a)`

But `dip` return is a function that has one argument with the same return type `a -> a`

```Haskell
inner-dip :: (a -> a -> a) -> a -> a
outer-dip :: (b -> b -> b) -> b -> b

outer-dip inner-dip

b = (a -> a -> a)
```
so `outer-dip` expectind `(b -> b -> b)` what is `(a -> a -> a) -> (a -> a -> a) -> a -> a -> a`

but was given `(a -> a -> a) -> a -> a`

In [ ]:
e :: (a -> a) -> a -> a
e = twice twice twice

e (+ 1) 0

16

Answer (e): `(a -> a) -> a -> a`


Intuition:
```
twice :: (a -> a) -> a -> a


twice f x = f (f x)

twice twice f x 
    = twice f (twice f x) 
    = twice f (f (f x))
    = (f (f (f (f x))))
```

Step-by-step:
```Haskell

twiceInner :: (a -> a) -> a -> a
twiceOuter :: (b -> b) -> b -> b
(b -> b) = (a -> a) -> a -> a

twiceOuter :: ((a -> a) -> (a -> a)) -> (a -> a) -> a -> a
twiceOuter twiceInner ::  (a -> a) -> a -> a
```
So `twiceOuter` applyed to `twiceInner` doesn't change it's type, so applying again will aslo not change the type

Task 3

In [ ]:
studentsWithB :: [Student] -> [Name]

studentsWithB [] = []
studentsWithB (Student name B : xs) = name : studentsWithB xs
studentsWithB (_ : xs) = studentsWithB xs

In [ ]:
studentsWithB [Student "Jack" C, Student "Jane" B]
-- ["Jane"]

["Jane"]

Task 4. (a)

In [ ]:
-- some magic code for printing Status a
instance Show a => Show (Status a) where
    show Stop         = "Stop"
    show (Warn msgs a) = "Warn " ++ show msgs ++ " " ++ show a

In [ ]:
filterName :: (Name -> Bool) -> [(Name, a)] -> [(Name, a)]
filterName _ [] = []

filterName f ((x, a):xs)
    | f x       = (x, a) : filterName f xs
    | otherwise = filterName f xs


apply :: (a -> b) -> [a] -> [b]
apply _ [] = []
apply f (x:xs) = f x : apply f xs

lookupName :: (Name -> Bool) -> [(Name, a)] -> Status a

lookupName f xs
    | null filtered        = Stop
    | length filtered == 1 = Warn [] (snd (head filtered))
    | length filtered > 1  = Warn (apply (\x -> "ignoring entry: " ++ fst x ) (tail filtered)) (snd (head filtered))
    where
        filtered = filterName f xs

In [ ]:
ages :: [(Name, Int)]
ages = [("John", 21), ("Jack", 23), ("Jane", 22), ("Jenny", 21)]


lookupName (== "Jack") ages

Warn [] 23

In [ ]:
lookupName (\name -> length name == 4) ages

Warn ["ignoring entry: Jack","ignoring entry: Jane"] 21

In [ ]:
lookupName (\name -> length name == 5) ages

Warn [] 21

In [ ]:
lookupName (\name -> length name > 5) ages

Stop

Task 4. (b)

In [ ]:
produceAll :: (a -> Status a) -> a -> [a]

produceAll f current
  | Warn _ next <- f current = current : produceAll f next
  | Stop        <- f current = []

In [ ]:
fizzbuzz :: Integer -> Status Integer
fizzbuzz n
    | fizz && buzz = Warn ["fizzbuzz"] m
    | fizz = Warn ["fizz"] m
    | buzz = Warn ["buzz"] m
    | otherwise = fizzbuzz m
    where
      m = n + 1
      fizz = m `mod` 3 == 0
      buzz = m `mod` 5 == 0

In [ ]:
take 10 (produceAll fizzbuzz 0)

[0,3,5,6,9,10,12,15,18,20]

In [ ]:
 produceAll (\n -> if n < 15 then fizzbuzz n else Stop) 0

[0,3,5,6,9,10,12]

Task 4. (c)

In [ ]:
produceAllWithWarnings :: (a -> Status a) -> a -> ([String], [a])
produceAllWithWarnings f current
  | Warn warn next <- f current =
      let (warns, results) = produceAllWithWarnings f next
      in (warn ++ warns, current : results)
  | Stop           <- f current = ([], [])

In [ ]:
produceAllWithWarnings (\n -> if n < 15 then fizzbuzz n else Stop) 0

(["fizz","buzz","fizz","fizz","buzz","fizz","fizzbuzz"],[0,3,5,6,9,10,12])

In [ ]:
take 7 (fst (produceAllWithWarnings fizzbuzz 0))

["fizz","buzz","fizz","fizz","buzz","fizz","fizzbuzz"]

Task 4. (d)

In [ ]:
applyStatus :: Status (a -> b) -> Status a -> Status b

applyStatus (Warn warns f) (Warn warnsA a) = Warn (warns ++ warnsA) (f a)   
applyStatus Stop _ = Stop
applyStatus _ Stop = Stop

In [ ]:
operators :: [(Name, Int -> Int)]
operators = [("inc", (+1)), ("double", (*2))]
constants :: [(Name, Int)]
constants = [("x", 3), ("y", 4), ("z", 7)]

In [ ]:
applyStatus (lookupName (== "double") operators) (lookupName (== "x") constants)

Warn [] 6

In [ ]:
applyStatus (lookupName (const True) operators) (lookupName (== "x") constants)

Warn ["ignoring entry: double"] 4

In [ ]:
applyStatus (lookupName (const True) operators) (lookupName (const True) constants)

Warn ["ignoring entry: double","ignoring entry: y","ignoring entry: z"] 4

In [ ]:
applyStatus (lookupName (== "triple") operators) (lookupName (== "x") constants)

Stop